# Прогноз дохода клиента — решение v3 (WMAE)

**Пост-мортем v2 (public 139 807 — хуже константного бейзлайна).** Причина найдена и воспроизводима: **утечка в leave-one-out target encoding**. LOO-код строки `(S − yᵢ + K·glob)/(n−1+K)` обратимо содержит её собственный таргет; бустинг декодировал yᵢ из признака (te-работодателя — топ-1 важности с 10-кратным отрывом), OOF всех моделей рухнул с 60k до 110k, а вырожденная калибровка (a=1.74) добила прогнозы на test. Анти-дрейф блок при этом отработал честно (тренд-множитель отвергнут: +23% на холдауте).

**Что изменено в v3:**

1. **TE без утечки**: train-часть фолда кодируется внутренним 5-fold OOF (таргет строки не попадает в её признак никаким путём); работодатель (ИНН) исключён из TE; добавлена диагностика стабильности TE по месяцам и покрытия test.
2. **Time-aware мета-слой**: веса ансамбля, аффинная и корзиночная калибровки подбираются на OOF последних 2 месяцев train (ближайший аналог test), границы калибровок зажаты (a∈[0.95;1.10], бины [0.90;1.15]).
3. **Тюнинг на форвард-холдауте** (обучение ранние месяцы -> проверка последние 2): оптимизируем то, что устроено как лидерборд; 1 фит на трейл — быстрее в 3 раза.
4. **Диверсификация класса моделей**: Ridge на log-доходе и LightGBM linear_tree (кусочно-линейные листья) — экстраполируют и ошибаются иначе, чем чистые деревья; выброшены dart и q40 (не дали ценности).
5. **Автоудаление train-only признаков** (например, first_salary_income: 11% в train, 0% в test).
6. **Санитарный контроль перед сабмитом**: квантили прогноза vs train и сравнение OOF с уровнем v1 — блок, которого не хватило, чтобы поймать v2 до отправки.

Порядок запуска: `FAST_MODE=True` (смоук ~2-3 мин) -> `FAST_MODE=False`, `PROFILE='STRONG'` (~10-15 ч, рекомендуется) или `'FULL'` (~20-30 ч). Кэш в `artifacts_v3/` — прерывание безопасно.


In [ ]:
# 0. КОНФИГУРАЦИЯ
FAST_MODE = False
PROFILE   = 'STRONG'

import os, re, json, time, random, warnings, hashlib, itertools
from pathlib import Path

from pandas.util import hash_pandas_object
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

def find_path(candidates, required=True):
    for p in candidates:
        if Path(p).exists():
            return p
    if required:
        raise FileNotFoundError(f'Не найден ни один из файлов: {candidates}')
    return None

TRAIN_PATH      = find_path(['train.csv', 'hackathon_income_train.csv', 'data/train.csv'])
TEST_PATH       = find_path(['test.csv', 'hackathon_income_test.csv', 'data/test.csv'])
DESC_PATH       = find_path(['feature_description.csv', 'data/feature_description.csv'], required=False)
SAMPLE_SUB_PATH = find_path(['sample_submission.csv', 'data/sample_submission.csv'], required=False)

SEED = 42
if FAST_MODE:
    SEEDS, N_FOLDS = [42], 3
    TRIALS = dict(lgb=2, xgb=1, cb=1)
elif PROFILE == 'FULL':
    SEEDS, N_FOLDS = [42, 777, 2026], 10
    TRIALS = dict(lgb=40, xgb=16, cb=8)
else:  # STRONG
    SEEDS, N_FOLDS = [42, 777], 5
    TRIALS = dict(lgb=30, xgb=12, cb=6)

N_THREADS = min(os.cpu_count() or 8, 16)

RUN_ADVERSARIAL  = True
RUN_TIME_HOLDOUT = True

ART_DIR = Path('artifacts_v3'); ART_DIR.mkdir(exist_ok=True)
CACHE_DIR = ART_DIR / 'cache'; CACHE_DIR.mkdir(exist_ok=True)
USE_CACHE = True
SUBMISSION_PATH = 'submission.csv'

random.seed(SEED); np.random.seed(SEED)
print(f'FAST_MODE={FAST_MODE} | PROFILE={PROFILE} | folds={N_FOLDS} | seeds={SEEDS} | threads={N_THREADS}')
print(f'Тюнинг Optuna (трейлов, 1 форвард-фит на трейл): {TRIALS} | кэш: {CACHE_DIR}')
print(f'train={TRAIN_PATH} | test={TEST_PATH} | desc={DESC_PATH} | sample_sub={SAMPLE_SUB_PATH}')


In [ ]:
# 1. МЕТРИКА
def wmae(y_true, y_pred, weights):
    y_true  = np.asarray(y_true, dtype=float)
    y_pred  = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

def weighted_median(values, weights):
    v = np.asarray(values, dtype=float); w = np.asarray(weights, dtype=float)
    order = np.argsort(v); v, w = v[order], w[order]
    cw = np.cumsum(w)
    return float(v[np.searchsorted(cw, 0.5 * cw[-1])])


In [ ]:
# 2. ЗАГРУЗКА ДАННЫХ
t0 = time.time()
train = pd.read_csv(TRAIN_PATH, sep=';', decimal=',')
test  = pd.read_csv(TEST_PATH,  sep=';', decimal=',')
print(f'train: {train.shape} | test: {test.shape} | загрузка {time.time()-t0:.1f}s')

assert {'target', 'w', 'id', 'dt'} <= set(train.columns), 'В train нет target/w/id/dt'
only_train = [c for c in train.columns if c not in test.columns]
print('Колонки только в train:', only_train)

dup_tr = int(train["id"].duplicated().sum()); dup_te = int(test["id"].duplicated().sum())
inter  = len(set(train["id"]) & set(test["id"]))
print(f'Дубликаты id: train={dup_tr}, test={dup_te} | пересечение id train∩test: {inter}')
USE_GROUPS = dup_tr > 0
print('Схема CV:', 'StratifiedGroupKFold по id' if USE_GROUPS else 'StratifiedKFold по децилям target')

DESC = {}
if DESC_PATH:
    for enc in ('cp1251', 'utf-8', 'utf-8-sig'):
        try:
            _d = pd.read_csv(DESC_PATH, sep=';', encoding=enc)
            DESC = dict(zip(_d.iloc[:, 0].astype(str), _d.iloc[:, 1].astype(str)))
            break
        except Exception:
            continue
print('Описаний признаков загружено:', len(DESC))


train: (76786, 224) | test: (73214, 222) | загрузка 1.9s
Колонки только в train: ['target', 'w']
Дубликаты id: train=0, test=0 | пересечение id train∩test: 0
Схема CV: StratifiedKFold по децилям target
Описаний признаков загружено: 224


In [ ]:
# 3. БЫСТРЫЙ EDA (материал для защиты)
y_full = train['target'].astype(float)
w_full = train['w'].astype(float)

print('--- target (доход, руб/мес) ---')
print(y_full.describe().round(1).to_string())
q = y_full.quantile([.01, .05, .25, .5, .75, .95, .99]).round(0)
print('Квантили:', q.to_dict())
print(f'Скошенность (skew): {y_full.skew():.2f}  -> сильно правый хвост')

print('\n--- веса w ---')
print(w_full.describe().round(4).to_string())
print(f'corr(w, target): pearson={np.corrcoef(w_full, y_full)[0,1]:.3f}')

print('\n--- даты среза dt ---')
def _parse_dt_head(s):
    out = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    if out.isna().mean() > 0.5:
        out = pd.to_datetime(s, format='mixed', dayfirst=True, errors='coerce')
    return out
_dt_tr = _parse_dt_head(train['dt'])
_dt_te = _parse_dt_head(test['dt'])
print('train:', _dt_tr.min().date(), '->', _dt_tr.max().date(),
      '| test:', _dt_te.min().date(), '->', _dt_te.max().date())
print('=> ВРЕМЕННОЙ сдвиг: test строго позже train — учитываем в валидации,'
      ' абсолютное время в признаки не подаём')
print('\nРаспределение по месяцам (train):')
print(_dt_tr.dt.to_period('M').value_counts().sort_index().to_string())

miss = train.isna().mean().sort_values(ascending=False)
print(f'\nПропуски: колонок с NaN>50%: {(miss > .5).sum()} | NaN>90%: {(miss > .9).sum()} '
      '(бустинги обрабатывают NaN нативно — не заполняем и не выбрасываем)')

_dtA = pd.to_datetime(train['dt'], format='%Y-%m-%d', errors='coerce')
if _dtA.isna().mean() > 0.5:
    _dtA = pd.to_datetime(train['dt'], format='mixed', dayfirst=True, errors='coerce')
_dtB = pd.to_datetime(test['dt'], format='%Y-%m-%d', errors='coerce')
if _dtB.isna().mean() > 0.5:
    _dtB = pd.to_datetime(test['dt'], format='mixed', dayfirst=True, errors='coerce')
_monA = (_dtA.dt.year * 12 + _dtA.dt.month).astype(int)

print('\n[v2] Взвешенная медиана target по месяцам train (тренд номинального дохода):')
_meds = {}
for _m in sorted(_monA.unique()):
    _ix = (_monA == _m).values
    _meds[_m] = weighted_median(train.loc[_ix, 'target'], train.loc[_ix, 'w'])
    print(f'  {(_m-1)//12}-{(_m-1)%12+1:02d}: {_meds[_m]:>10,.0f}')
_ms = np.array(sorted(_meds)); _beta = np.polyfit(_ms - _ms.min(), np.log([_meds[m] for m in _ms]), 1)[0]
TREND_G_RAW = float(np.exp(_beta) - 1)
print(f'[v2] Оценка роста медианного дохода: {TREND_G_RAW*100:+.2f}%/мес '
      f'(test отстоит от последнего train-месяца на 1-5 мес)')

print('\n[v2] Средний вес w по месяцам train:')
print(train.groupby(_monA)['w'].mean().round(4).to_string())

_key_sal = [c for c in ['salary_6to12m_avg', 'dp_ils_avg_salary_1y', 'first_salary_income',
                        'incomeValue'] if c in train.columns]
print('\n[v2] Заполненность ключевых зарплатных полей, train vs test:')
for _c in _key_sal:
    print(f'  {_c:32s} train={pd.to_numeric(train[_c], errors="coerce").notna().mean():.3f}'
          f'  test={pd.to_numeric(test[_c], errors="coerce").notna().mean():.3f}')
if 'per_capita_income_rur_amt' in train.columns:
    print('\n[v2] Медиана per_capita_income_rur_amt по месяцам (train, затем test):')
    _pc_tr = pd.to_numeric(train['per_capita_income_rur_amt'], errors='coerce')
    _pc_te = pd.to_numeric(test['per_capita_income_rur_amt'], errors='coerce')
    _monB = (_dtB.dt.year * 12 + _dtB.dt.month).astype(int)
    _s = pd.concat([_pc_tr.groupby(_monA).median(), _pc_te.groupby(_monB).median()])
    print(_s.round(0).to_string())


In [ ]:
# 4. ТИПИЗАЦИЯ И ОЧИСТКА КОЛОНОК
ID_COL, TARGET_COL, W_COL = 'id', 'target', 'w'
DATE_COLS  = ['dt', 'period_last_act_ad']
KNOWN_CATS = ['gender', 'adminarea', 'city_smart_name', 'addrref',
              'dp_ewb_last_employment_position', 'dp_ewb_last_organization']

def is_nonnumeric(s):
    return not (pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s))

def try_numeric(s):
    conv = pd.to_numeric(s.astype('string').str.replace(',', '.', regex=False), errors='coerce')
    ok = conv.notna().sum() >= 0.99 * max(int(s.notna().sum()), 1)
    return conv.astype('float64'), ok

fixed, cat_cols = [], []
for c in train.columns:
    if c in (ID_COL, TARGET_COL, W_COL) or c in DATE_COLS:
        continue
    in_test = c in test.columns
    nonnum = is_nonnumeric(train[c]) or (in_test and is_nonnumeric(test[c]))
    if nonnum and c not in KNOWN_CATS:
        both = pd.concat([train[c], test[c]]) if in_test else train[c]
        conv, ok = try_numeric(both)
        if ok:
            train[c] = conv.iloc[:len(train)].values
            if in_test:
                test[c] = conv.iloc[len(train):].values
            fixed.append(c)
        else:
            cat_cols.append(c)
    elif c in KNOWN_CATS:
        cat_cols.append(c)
print(f'Строковых колонок восстановлено в числа: {len(fixed)}')
print('Категориальные признаки:', cat_cols)

def parse_dt(s):
    s = s.astype('string').str.strip()
    out = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    mask = out.isna()
    if mask.any():
        out.loc[mask] = pd.to_datetime(s[mask], format='%d.%m.%Y', errors='coerce')
    mask = out.isna()
    if mask.any():
        out.loc[mask] = pd.to_datetime(s[mask], format='mixed', dayfirst=True, errors='coerce')
    return out

for df in (train, test):
    df['dt_parsed'] = parse_dt(df['dt'])
    if 'period_last_act_ad' in df.columns:
        df['pla_parsed'] = parse_dt(df['period_last_act_ad'])

def clean_position(s):
    s = s.astype('string').str.lower()
    s = s.str.replace(r'[\d]+', ' ', regex=True)
    s = s.str.replace(r'[^а-яёa-z ]+', ' ', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s.replace('', pd.NA)

for df in (train, test):
    if 'dp_ewb_last_organization' in df.columns:
        df['dp_ewb_last_organization'] = df['dp_ewb_last_organization'].astype('Int64').astype('string')
    if 'dp_ewb_last_employment_position' in df.columns:
        df['dp_ewb_last_employment_position'] = clean_position(df['dp_ewb_last_employment_position'])

base_cols = [c for c in train.columns
             if c not in (ID_COL, TARGET_COL, W_COL, 'dt', 'dt_parsed', 'period_last_act_ad', 'pla_parsed')
             and c in test.columns]
const_cols = [c for c in base_cols if train[c].nunique(dropna=False) <= 1]

comb = pd.concat([train[base_cols], test[base_cols]], ignore_index=True)
sig = {}
dup_cols = []
for c in base_cols:
    if c in const_cols:
        continue
    h = int(hash_pandas_object(
        comb[c].astype('string') if is_nonnumeric(comb[c]) else comb[c],
        index=False).sum())
    if h in sig and comb[c].equals(comb[sig[h]]):
        dup_cols.append(c)
    else:
        sig[h] = c
del comb
print(f'Удалено константных: {len(const_cols)} {const_cols[:5]}; дублей: {len(dup_cols)} {dup_cols[:5]}')
DROP_COLS = set(const_cols) | set(dup_cols)
cat_cols  = [c for c in cat_cols if c not in DROP_COLS]


Строковых колонок восстановлено в числа: 34
Категориальные признаки: ['gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref', 'dp_ewb_last_organization']
Удалено константных: 0 []; дублей: 0 []


In [ ]:
# 5. FEATURE ENGINEERING
def sdiv(a, b):
    a = pd.to_numeric(a, errors='coerce'); b = pd.to_numeric(b, errors='coerce')
    out = a / b.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)

SALARY_SRC = [c for c in ['salary_6to12m_avg', 'dp_ils_avg_salary_1y', 'dp_ils_avg_salary_2y',
                          'dp_ils_avg_salary_3y', 'first_salary_income', 'incomeValue',
                          'dp_payoutincomedata_payout_avg_3_month',
                          'dp_payoutincomedata_payout_avg_6_month'] if c in train.columns]

def add_features(df):
    f = {}
    sal = df[SALARY_SRC].apply(pd.to_numeric, errors='coerce')
    f['fe_salary_mean']   = sal.mean(axis=1)
    f['fe_salary_max']    = sal.max(axis=1)
    f['fe_salary_median'] = sal.median(axis=1)
    f['fe_salary_srccnt'] = sal.notna().sum(axis=1).astype(float)
    if {'salary_6to12m_avg', 'dp_ils_avg_salary_1y'} <= set(df.columns):
        f['fe_salary_bank_vs_dp'] = sdiv(df['salary_6to12m_avg'], df['dp_ils_avg_salary_1y'])
    if {'dp_ils_avg_salary_1y', 'dp_ils_avg_salary_3y'} <= set(df.columns):
        f['fe_salary_growth_1y3y'] = sdiv(df['dp_ils_avg_salary_1y'], df['dp_ils_avg_salary_3y'])
    for reg in ['per_capita_income_rur_amt', 'salary_median_in_gex_r1']:
        if reg in df.columns:
            f[f'fe_salary_vs_{reg[:14]}'] = sdiv(f['fe_salary_mean'], df[reg])
            if 'incomeValue' in df.columns:
                f[f'fe_incomeval_vs_{reg[:14]}'] = sdiv(df['incomeValue'], df[reg])
    if {'avg_cur_cr_turn', 'avg_cur_db_turn'} <= set(df.columns):
        f['fe_crdb_ratio_3m'] = sdiv(df['avg_cur_cr_turn'], df['avg_cur_db_turn'])
    if {'turn_cur_cr_avg_v2', 'turn_cur_db_avg_v2'} <= set(df.columns):
        f['fe_crdb_ratio_12m'] = sdiv(df['turn_cur_cr_avg_v2'], df['turn_cur_db_avg_v2'])
    if {'avg_cur_cr_turn', 'turn_cur_cr_avg_v2'} <= set(df.columns):
        f['fe_cr_trend_3m12m'] = sdiv(df['avg_cur_cr_turn'], df['turn_cur_cr_avg_v2'])
    if {'avg_cur_db_turn', 'turn_cur_db_avg_v2'} <= set(df.columns):
        f['fe_db_trend_3m12m'] = sdiv(df['avg_cur_db_turn'], df['turn_cur_db_avg_v2'])
    if 'turn_cur_cr_avg_v2' in df.columns:
        f['fe_turn_vs_salary'] = sdiv(df['turn_cur_cr_avg_v2'], f['fe_salary_mean'])
    if {'curr_rur_amt_cm_avg', 'turn_cur_db_avg_v2'} <= set(df.columns):
        f['fe_balance_vs_dbturn'] = sdiv(df['curr_rur_amt_cm_avg'], df['turn_cur_db_avg_v2'])
    if {'avg_3m_all', 'avg_6m_all'} <= set(df.columns):
        f['fe_spend_trend'] = sdiv(df['avg_3m_all'], df['avg_6m_all'])
    if {'avg_6m_all', 'avg_cur_db_turn'} <= set(df.columns):
        f['fe_spend_share_of_db'] = sdiv(df['avg_6m_all'], df['avg_cur_db_turn'])
    if {'hdb_outstand_sum', 'hdb_bki_total_max_limit'} <= set(df.columns):
        f['fe_bki_utilization'] = sdiv(df['hdb_outstand_sum'], df['hdb_bki_total_max_limit'])
    if {'hdb_bki_total_max_limit', 'hdb_bki_total_products'} <= set(df.columns):
        f['fe_bki_limit_per_prod'] = sdiv(df['hdb_bki_total_max_limit'], df['hdb_bki_total_products'])
    if 'loan_cur_amt' in df.columns:
        f['fe_loanreq_vs_salary'] = sdiv(df['loan_cur_amt'], f['fe_salary_mean'])
    ov = [c for c in ['ovrd_sum', 'hdb_ovrd_sum', 'total_sum'] if c in df.columns]
    if ov:
        f['fe_has_overdue'] = (df[ov].fillna(0) > 0).any(axis=1).astype(float)
    f['fe_nan_total'] = df.isna().sum(axis=1).astype(float)
    dp_cols  = [c for c in df.columns if c.startswith('dp_')]
    bki_cols = [c for c in df.columns if 'bki' in c or c.startswith('hdb_')]
    f['fe_nan_dp']  = df[dp_cols].isna().sum(axis=1).astype(float)  if dp_cols  else 0.0
    f['fe_nan_bki'] = df[bki_cols].isna().sum(axis=1).astype(float) if bki_cols else 0.0
    if 'pla_parsed' in df.columns:
        f['fe_days_since_last_act'] = (df['dt_parsed'] - df['pla_parsed']).dt.days.astype(float)
    return pd.DataFrame(f, index=df.index)

fe_train = add_features(train); fe_test = add_features(test)
train = pd.concat([train, fe_train], axis=1); test = pd.concat([test, fe_test], axis=1)
FE_NUM = list(fe_train.columns)
print(f'Создано инженерных признаков: {len(FE_NUM)}')

for c in cat_cols:
    vc = pd.concat([train[c], test[c]]).astype('string').value_counts()
    train[f'fe_freq_{c}'] = train[c].astype('string').map(vc).astype(float)
    test[f'fe_freq_{c}']  = test[c].astype('string').map(vc).astype(float)
    FE_NUM.append(f'fe_freq_{c}')

POS_KW = {
    'dir': r'директор|ген\W*дир', 'zam': r'\bзам\b|замест', 'head': r'начальник|руковод|заведующ',
    'chief': r'главн', 'lead': r'ведущ', 'senior': r'старш', 'spec': r'специалист',
    'manager': r'менеджер', 'engineer': r'инженер',
    'it': r'программист|разработ|аналитик|\bit\b|айти',
    'driver': r'водител|машинист', 'sales': r'продав|кассир',
    'accountant': r'бухгалтер|экономист|финанс',
    'doctor': r'врач|фельдшер|медицин|медсест', 'teacher': r'учител|преподават|воспитат|педагог',
    'worker': r'слесар|электрик|монтаж|свар|токар|груз|рабоч|уборщ|повар|охран|курьер|комплектов',
    'lawyer': r'юрист|юрискон', 'operator': r'оператор|диспетчер',
    'admin': r'администратор|секретар', 'master': r'мастер|бригадир|прораб',
    'consult': r'консультант',
}
SENIOR_W = {'dir': 3.0, 'zam': 2.4, 'head': 2.4, 'chief': 2.0, 'lead': 1.5, 'senior': 1.2,
            'master': 1.2, 'spec': 1.0, 'manager': 1.0, 'engineer': 1.1, 'it': 1.3,
            'accountant': 1.0, 'lawyer': 1.1, 'doctor': 1.1, 'teacher': 0.9, 'consult': 0.9,
            'operator': 0.8, 'admin': 0.8, 'sales': 0.7, 'driver': 0.8, 'worker': 0.7}
_pcol = 'dp_ewb_last_employment_position'
if _pcol in train.columns:
    for _df in (train, test):
        _s = _df[_pcol].astype('string').fillna('')
        _grade = pd.Series(np.nan, index=_df.index, dtype=float)
        for _k, _pat in POS_KW.items():
            _hit = _s.str.contains(_pat, regex=True, na=False)
            _df[f'fe_pos_{_k}'] = _hit.astype(float)
            _grade = np.fmax(_grade, np.where(_hit, SENIOR_W[_k], np.nan))
        _df['fe_pos_grade'] = _grade
        _df['fe_pos_len'] = _s.str.len().replace(0, np.nan).astype(float)
    FE_NUM += [f'fe_pos_{k}' for k in POS_KW] + ['fe_pos_grade', 'fe_pos_len']

def _add_v2(df):
    g = {}
    if {'dp_ils_avg_salary_1y', 'dp_ils_avg_salary_2y'} <= set(df.columns):
        g['fe_sal_growth_1y2y'] = sdiv(df['dp_ils_avg_salary_1y'], df['dp_ils_avg_salary_2y'])
    if {'dp_ils_avg_salary_2y', 'dp_ils_avg_salary_3y'} <= set(df.columns):
        g['fe_sal_growth_2y3y'] = sdiv(df['dp_ils_avg_salary_2y'], df['dp_ils_avg_salary_3y'])
    if 'first_salary_income' in df.columns:
        g['fe_first_vs_mean'] = sdiv(df['first_salary_income'], df['fe_salary_mean'])
    if 'hdb_bki_total_max_limit' in df.columns:
        g['fe_bkilim_vs_salary'] = sdiv(df['hdb_bki_total_max_limit'], df['fe_salary_mean'])
    if 'hdb_bki_total_cc_max_limit' in df.columns:
        g['fe_cclim_vs_salary'] = sdiv(df['hdb_bki_total_cc_max_limit'], df['fe_salary_mean'])
    if 'turn_cur_db_avg_v2' in df.columns:
        g['fe_dbturn_vs_salary'] = sdiv(df['turn_cur_db_avg_v2'], df['fe_salary_mean'])
    if 'age' in df.columns:
        g['fe_salary_per_age'] = sdiv(df['fe_salary_mean'], df['age'])
    return pd.DataFrame(g, index=df.index)

_a2 = _add_v2(train); _b2 = _add_v2(test)
train = pd.concat([train, _a2], axis=1); test = pd.concat([test, _b2], axis=1)
FE_NUM += list(_a2.columns)
print(f'[v2] Доп. признаки: должность={len(POS_KW)+2 if _pcol in train.columns else 0}, '
      f'динамика/отношения={len(_a2.columns)}')

EXCLUDE = {ID_COL, TARGET_COL, W_COL, 'dt', 'dt_parsed', 'period_last_act_ad', 'pla_parsed'} | DROP_COLS
FEATURES = [c for c in train.columns if c not in EXCLUDE and c in test.columns]
CAT_FEATURES = [c for c in cat_cols if c in FEATURES]
print(f'Всего признаков: {len(FEATURES)} (категориальных: {len(CAT_FEATURES)})')

SAFE = {c: re.sub(r'[^0-9a-zA-Z_]', '_', c) for c in FEATURES}
assert len(set(SAFE.values())) == len(SAFE)
INV_SAFE = {v: k for k, v in SAFE.items()}

X  = train[FEATURES].rename(columns=SAFE).copy()
Xt = test[FEATURES].rename(columns=SAFE).copy()
CATS = [SAFE[c] for c in CAT_FEATURES]
for c in CATS:
    cats_union = pd.unique(pd.concat([X[c], Xt[c]]).dropna().astype(str))
    ctype = pd.CategoricalDtype(categories=sorted(cats_union))
    X[c]  = X[c].astype(str).where(X[c].notna(), np.nan).astype(ctype)
    Xt[c] = Xt[c].astype(str).where(Xt[c].notna(), np.nan).astype(ctype)

X_cb, Xt_cb = X.copy(), Xt.copy()
for c in CATS:
    X_cb[c]  = X_cb[c].astype(object).where(X_cb[c].notna(), '__NA__').astype(str)
    Xt_cb[c] = Xt_cb[c].astype(object).where(Xt_cb[c].notna(), '__NA__').astype(str)

y = train[TARGET_COL].astype(float).values
w = train[W_COL].astype(float).values
print('Матрицы готовы:', X.shape, Xt.shape)


In [ ]:
# 6. ADVERSARIAL VALIDATION
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

Xa = pd.concat([X, Xt], ignore_index=True)
ya = np.r_[np.zeros(len(X)), np.ones(len(Xt))]
adv_par = dict(n_estimators=40 if FAST_MODE else 400, learning_rate=0.05, num_leaves=63,
               subsample=0.9, colsample_bytree=0.7, n_jobs=N_THREADS, verbosity=-1)
adv_oof = np.zeros(len(Xa)); adv_imp = np.zeros(X.shape[1])
for tr_i, va_i in StratifiedKFold(4, shuffle=True, random_state=SEED).split(Xa, ya):
    _m = lgb.LGBMClassifier(**adv_par, random_state=SEED)
    _m.fit(Xa.iloc[tr_i], ya[tr_i])
    adv_oof[va_i] = _m.predict_proba(Xa.iloc[va_i])[:, 1]
    adv_imp += _m.booster_.feature_importance('gain')
AUC_ADV = roc_auc_score(ya, adv_oof)
print(f'Adversarial AUC (OOF) = {AUC_ADV:.3f}  (0.5 = train и test неразличимы)')

_impser = pd.Series(adv_imp, index=X.columns).sort_values(ascending=False)
ADV_TOP_NUM = [c for c in _impser.index if c not in CATS][:20]
print('\nПризнаки, сильнее всего отличающие test от train (дрейф):')
for _name in _impser.index[:10]:
    _orig = INV_SAFE.get(_name, _name)
    print(f'  {_orig:55s} gain={_impser[_name]:12.0f}  | {DESC.get(_orig, "")[:70]}')

_p_tr = np.clip(adv_oof[:len(X)], 1e-3, 1 - 1e-3)
ADV_ODDS = _p_tr / (1 - _p_tr)
del Xa


In [ ]:
# 7. ВРЕМЕННОЙ ХОЛДАУТ + АНТИ-ДРЕЙФ ЭКСПЕРИМЕНТЫ
_fill_tr = X.notna().mean(); _fill_te = Xt.notna().mean()
TRAIN_ONLY_COLS = [c for c in X.columns if c not in CATS
                   and _fill_tr[c] > 0.02 and _fill_te[c] < 0.001]
if TRAIN_ONLY_COLS:
    print(f'[v3] Удаляю {len(TRAIN_ONLY_COLS)} train-only признаков (в test пусто):')
    for c in TRAIN_ONLY_COLS[:15]:
        print('   ', INV_SAFE.get(c, c))
    X, Xt = X.drop(columns=TRAIN_ONLY_COLS), Xt.drop(columns=TRAIN_ONLY_COLS)
    X_cb, Xt_cb = X_cb.drop(columns=TRAIN_ONLY_COLS), Xt_cb.drop(columns=TRAIN_ONLY_COLS)

import lightgbm as lgb

MON_TR = (train['dt_parsed'].dt.year * 12 + train['dt_parsed'].dt.month).astype(int).values
MON_TE = (test['dt_parsed'].dt.year * 12 + test['dt_parsed'].dt.month).astype(int).values
MON_TR_S = pd.Series(MON_TR, index=X.index)
MON_TE_S = pd.Series(MON_TE, index=Xt.index)

_mmax = MON_TR.max()
hmask_va = MON_TR >= _mmax - 1
hmask_tr = ~hmask_va
print(f'Holdout: обучение {hmask_tr.sum()} строк (ранние месяцы) -> валидация {hmask_va.sum()} строк (последние 2 мес)')

def fit_trend_g(mon_int, yv, wv):
    ms = np.unique(mon_int)
    if len(ms) < 3:
        return 0.0
    med = np.array([weighted_median(yv[mon_int == m0], wv[mon_int == m0]) for m0 in ms])
    beta = np.polyfit(ms - ms.min(), np.log(np.maximum(med, 1.0)), 1)[0]
    return float(np.exp(beta) - 1)

def month_rank(df_x, mon_s, cols):
    out = df_x.copy()
    for c in cols:
        out[c] = out[c].groupby(mon_s).rank(pct=True)
    return out

EXP_PAR = dict(objective='l1',
               n_estimators=200 if FAST_MODE else 5000,
               learning_rate=0.3 if FAST_MODE else 0.05,
               num_leaves=15 if FAST_MODE else 96,
               min_child_samples=5 if FAST_MODE else 60,
               colsample_bytree=0.7, subsample=0.85, subsample_freq=1,
               reg_lambda=5.0, n_jobs=N_THREADS, force_row_wise=True, verbosity=-1)

def hold_wmae(drop=(), rank_cols=(), adv_w=False, trend=False):
    Xa_ = X[hmask_tr]; Xb_ = X[hmask_va]
    if rank_cols:
        Xa_ = month_rank(Xa_, MON_TR_S[hmask_tr], rank_cols)
        Xb_ = month_rank(Xb_, MON_TR_S[hmask_va], rank_cols)
    if drop:
        Xa_ = Xa_.drop(columns=list(drop)); Xb_ = Xb_.drop(columns=list(drop))
    wtr = w[hmask_tr].copy()
    if adv_w:
        mlt = np.clip(ADV_ODDS[hmask_tr], 0.25, 4.0)
        wtr = wtr * mlt * (wtr.sum() / (wtr * mlt).sum())
    m = lgb.LGBMRegressor(**EXP_PAR, random_state=SEED)
    m.fit(Xa_, y[hmask_tr], sample_weight=wtr,
          eval_set=[(Xb_, y[hmask_va])], eval_sample_weight=[w[hmask_va]], eval_metric='l1',
          callbacks=[lgb.early_stopping(20 if FAST_MODE else 150, verbose=False)])
    p = m.predict(Xb_)
    if trend:
        g_ = fit_trend_g(MON_TR[hmask_tr], y[hmask_tr], w[hmask_tr])
        p = p * (1.0 + g_) ** (MON_TR[hmask_va] - MON_TR[hmask_tr].max())
    return wmae(y[hmask_va], p, w[hmask_va])

t0_exp = time.time()
DRIFT_K = 8
DROP_CAND = [c for c in ADV_TOP_NUM[:DRIFT_K] if c in X.columns]
RES_EXP = {'E0': hold_wmae(),
           'E1': hold_wmae(drop=DROP_CAND),
           'E2': hold_wmae(rank_cols=DROP_CAND),
           'E3': hold_wmae(adv_w=True),
           'E4': hold_wmae(trend=True)}
_names = {'E0': 'база', 'E1': f'drop {DRIFT_K} дрейфующих', 'E2': 'rank-в-месяце дрейфующих',
          'E3': 'adversarial-веса', 'E4': 'тренд-множитель по месяцам'}
_base = RES_EXP['E0']
for k_, v_ in RES_EXP.items():
    print(f'  {k_} {_names[k_]:28s} WMAE={v_:10.2f}  ({(v_/_base-1)*100:+.2f}%)')

_improved = [e for e in ('E1', 'E2', 'E3', 'E4') if RES_EXP[e] < _base * 0.999]
best_combo, best_v_ = (), _base
for r_ in range(1, len(_improved) + 1):
    for comb in itertools.combinations(_improved, r_):
        if 'E1' in comb and 'E2' in comb:
            continue
        if len(comb) == 1:
            v_ = RES_EXP[comb[0]]
        else:
            v_ = hold_wmae(drop=DROP_CAND if 'E1' in comb else (),
                           rank_cols=DROP_CAND if 'E2' in comb else (),
                           adv_w='E3' in comb, trend='E4' in comb)
            print(f'  комбо {"+".join(comb):20s} WMAE={v_:10.2f}  ({(v_/_base-1)*100:+.2f}%)')
        if v_ < best_v_:
            best_v_, best_combo = v_, comb

DEC = dict(drop=list(DROP_CAND) if 'E1' in best_combo else [],
           rank=list(DROP_CAND) if 'E2' in best_combo else [],
           adv_w=('E3' in best_combo), trend_g=0.0)
if 'E4' in best_combo:
    DEC['trend_g'] = fit_trend_g(MON_TR, y, w)
print(f'\nПринятая комбинация: {best_combo or "ничего (оставляем базу)"} | '
      f'holdout WMAE {_base:.2f} -> {best_v_:.2f} | время {time.time()-t0_exp:.0f}s')
print('Решения анти-дрейфа:', json.dumps({k: (v if not isinstance(v, list) else f"{len(v)} колонок")
                                          for k, v in DEC.items()}, ensure_ascii=False))

if DEC['rank']:
    X    = month_rank(X,    MON_TR_S, DEC['rank']);  Xt    = month_rank(Xt,    MON_TE_S, DEC['rank'])
    X_cb = month_rank(X_cb, MON_TR_S, DEC['rank']);  Xt_cb = month_rank(Xt_cb, MON_TE_S, DEC['rank'])
if DEC['drop']:
    X, Xt   = X.drop(columns=DEC['drop']),    Xt.drop(columns=DEC['drop'])
    X_cb, Xt_cb = X_cb.drop(columns=DEC['drop']), Xt_cb.drop(columns=DEC['drop'])

W_TRAIN = w.copy()
if DEC['adv_w']:
    _mlt = np.clip(ADV_ODDS, 0.25, 4.0)
    W_TRAIN = w * _mlt * (w.sum() / (w * _mlt).sum())

TEST_TREND_MULT = (1.0 + DEC['trend_g']) ** np.maximum(MON_TE - MON_TR.max(), 0)
print(f'Матрицы после анти-дрейфа: {X.shape} / {Xt.shape} | '
      f'trend-множители test: {TEST_TREND_MULT.min():.3f}..{TEST_TREND_MULT.max():.3f}')


In [ ]:
# 8. СХЕМА КРОСС-ВАЛИДАЦИИ
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
target_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

def make_splits(seed):
    if USE_GROUPS:
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        return list(cv.split(X, target_bins, groups=train[ID_COL]))
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return list(cv.split(X, target_bins))

SPLITS = {s: make_splits(s) for s in SEEDS}
print({s: [len(v) for _, v in sp] for s, sp in SPLITS.items()})


{42: [15358, 15357, 15357, 15357, 15357], 777: [15358, 15357, 15357, 15357, 15357]}


In [ ]:
# 8b. TARGET ENCODING v3 (inner-OOF, БЕЗ утечки)
from sklearn.model_selection import KFold as _KFold

TE_SRC = [c for c in ['city_smart_name', 'adminarea', 'addrref',
                      'dp_ewb_last_employment_position'] if c in CAT_FEATURES]
TE_KEYS   = {c: train[c].astype('string').fillna('__NA__').values for c in TE_SRC}
TE_KEYS_T = {c: test[c].astype('string').fillna('__NA__').values for c in TE_SRC}
TE_K = 50.0
_ylog = np.log1p(y)
TE_GLOB = float(_ylog.mean())
TE_COLS_OUT = [f'te_{SAFE[c]}' for c in TE_SRC]

def _te_maps(idx):
    maps = {}
    for c in TE_SRC:
        g = pd.DataFrame({'k': TE_KEYS[c][idx], 'v': _ylog[idx]}).groupby('k')['v'].agg(['sum', 'count'])
        maps[c] = (g['sum'] + TE_GLOB * TE_K) / (g['count'] + TE_K)
    return maps

def _te_apply(keys_dict, idx, maps, index_out):
    data = {}
    for c in TE_SRC:
        data[f'te_{SAFE[c]}'] = pd.Series(keys_dict[c][idx]).map(maps[c]).fillna(TE_GLOB).values
    return pd.DataFrame(data, index=index_out)

def _te_inner_oof(tr_i):
    out = np.full((len(tr_i), len(TE_SRC)), TE_GLOB)
    for itr, iva in _KFold(5, shuffle=True, random_state=SEED).split(tr_i):
        maps = _te_maps(tr_i[itr])
        for j, c in enumerate(TE_SRC):
            out[iva, j] = pd.Series(TE_KEYS[c][tr_i[iva]]).map(maps[c]).fillna(TE_GLOB).values
    return pd.DataFrame(out, columns=TE_COLS_OUT, index=X.index[tr_i])

_TE_CACHE = {}
def te_for_pair(tr_i, va_i, key):
    if key not in _TE_CACHE:
        maps = _te_maps(tr_i)
        _TE_CACHE[key] = (_te_inner_oof(tr_i),
                          _te_apply(TE_KEYS, va_i, maps, X.index[va_i]))
    return _TE_CACHE[key]

def te_for_fold(seed, k, tr_i, va_i):
    return te_for_pair(tr_i, va_i, key=(seed, k))

TE_FULL_MAPS = _te_maps(np.arange(len(y)))
TE_TEST = _te_apply(TE_KEYS_T, np.arange(len(Xt)), TE_FULL_MAPS, Xt.index)
print(f'Target encoding v3: {TE_SRC} -> {len(TE_COLS_OUT)} колонок (inner-OOF, K={TE_K:.0f})')

_mon_all = (train['dt_parsed'].dt.year * 12 + train['dt_parsed'].dt.month).astype(int).values
_early = np.where(_mon_all <= np.median(_mon_all))[0]; _late = np.where(_mon_all > np.median(_mon_all))[0]
print('Диагностика TE: corr(ранние месяцы, поздние месяцы) по общим ключам | % неизвестных ключей в test:')
_m_early, _m_late = _te_maps(_early), _te_maps(_late)
for c in TE_SRC:
    m1, m2 = _m_early[c], _m_late[c]
    common = m1.index.intersection(m2.index)
    corr = float(np.corrcoef(m1[common], m2[common])[0, 1]) if len(common) > 10 else float('nan')
    unseen = float((~pd.Series(TE_KEYS_T[c]).isin(TE_FULL_MAPS[c].index)).mean())
    print(f'  {c:38s} corr={corr:.3f} | unseen_test={unseen:.1%}')


In [ ]:
# 9. ОБУЧАЮЩИЙ КАРКАС v3 (OOF + дисковый кэш)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

ES = 15 if FAST_MODE else 400
NE = 60 if FAST_MODE else 30000

NUM_COLS = [c for c in X.columns if c not in CATS]
LATE_MASK = hmask_va.copy()
TXT_COL = 'dp_ewb_last_employment_position'
TXT_TR = train[TXT_COL].astype('string').fillna('') if TXT_COL in train.columns else None
TXT_TE = test[TXT_COL].astype('string').fillna('')  if TXT_COL in test.columns  else None

LGB_D = dict(objective='l1', n_estimators=NE, learning_rate=0.2 if FAST_MODE else 0.03,
             num_leaves=31 if FAST_MODE else 127, min_child_samples=10 if FAST_MODE else 60,
             colsample_bytree=0.65, subsample=0.85, subsample_freq=1,
             reg_lambda=5.0, reg_alpha=0.5, max_bin=255,
             n_jobs=N_THREADS, force_row_wise=True, verbosity=-1)
XGB_D = dict(objective='reg:absoluteerror', eval_metric='mae', n_estimators=NE,
             learning_rate=0.2 if FAST_MODE else 0.03, max_depth=6 if FAST_MODE else 8,
             min_child_weight=10, subsample=0.85, colsample_bytree=0.65,
             reg_lambda=5.0, reg_alpha=0.5, tree_method='hist',
             enable_categorical=True, max_cat_to_onehot=1,
             early_stopping_rounds=ES, n_jobs=N_THREADS)
CB_D = dict(loss_function='MAE', iterations=80 if FAST_MODE else 22000,
            learning_rate=0.2 if FAST_MODE else 0.05, depth=6 if FAST_MODE else 8,
            l2_leaf_reg=6.0, bootstrap_type='Bernoulli', subsample=0.8, rsm=0.8,
            od_type='Iter', od_wait=ES, thread_count=N_THREADS,
            allow_writing_files=False, verbose=False)
HGB_D = dict(loss='absolute_error', max_iter=80 if FAST_MODE else 2500,
             learning_rate=0.3 if FAST_MODE else 0.05,
             max_leaf_nodes=15 if FAST_MODE else 96, min_samples_leaf=40,
             l2_regularization=1.0, max_bins=255,
             early_stopping=True, validation_fraction=0.08, n_iter_no_change=60)

def _cfg_hash(*parts):
    return hashlib.sha1(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:14]

DEC_SIG = _cfg_hash('v3', DEC, N_FOLDS, sorted(map(str, X.columns)), TE_COLS_OUT)

def _lin_matrix(df, med, mu=None, sd=None):
    v = df.fillna(med).fillna(0.0).to_numpy(dtype=float)
    v = np.sign(v) * np.log1p(np.abs(v))
    if mu is None:
        mu, sd = v.mean(axis=0), v.std(axis=0) + 1e-9
    return (v - mu) / sd, mu, sd

def fit_predict(engine, params, seed, Xtr, ytr, wtr, Xva, yva, wva, Xte, cats, txt):
    if engine == 'lgb':
        m = lgb.LGBMRegressor(**params, random_state=seed)
        cbs = [] if params.get('boosting_type') == 'dart' else [lgb.early_stopping(ES, verbose=False)]
        emetric = 'quantile' if params.get('objective') == 'quantile' else 'l1'
        m.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)], eval_sample_weight=[wva],
              eval_metric=emetric, callbacks=cbs)
        return m.predict(Xva), m.predict(Xte), m
    if engine == 'xgb':
        m = xgb.XGBRegressor(**params, random_state=seed)
        m.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)],
              sample_weight_eval_set=[wva], verbose=False)
        return m.predict(Xva), m.predict(Xte), m
    if engine == 'cb':
        m = CatBoostRegressor(**params, random_seed=seed)
        m.fit(Pool(Xtr, ytr, weight=wtr, cat_features=cats, text_features=txt),
              eval_set=Pool(Xva, yva, weight=wva, cat_features=cats, text_features=txt),
              use_best_model=True)
        return (m.predict(Pool(Xva, cat_features=cats, text_features=txt)),
                m.predict(Pool(Xte, cat_features=cats, text_features=txt)), m)
    if engine == 'hgb':
        m = HistGradientBoostingRegressor(**params, random_state=seed)
        m.fit(Xtr, ytr, sample_weight=wtr)
        return m.predict(Xva), m.predict(Xte), m
    if engine == 'ridge':
        med = Xtr.median(numeric_only=True)
        A, mu, sd = _lin_matrix(Xtr, med)
        B, _, _ = _lin_matrix(Xva, med, mu, sd)
        C, _, _ = _lin_matrix(Xte, med, mu, sd)
        m = Ridge(alpha=params.get('alpha', 30.0), random_state=seed)
        m.fit(A, ytr, sample_weight=wtr)
        return m.predict(B), m.predict(C), m
    raise ValueError(engine)

def run_model(name, engine, params, log_target=False, use_cb=False, use_text=False,
              numeric_only=False, collect_imp=False, seeds=None):
    t0 = time.time(); seeds = seeds or SEEDS
    base_X, base_T = (X_cb, Xt_cb) if use_cb else (X, Xt)
    if numeric_only:
        base_X, base_T = X[NUM_COLS], Xt[NUM_COLS]
    cats = CATS if (use_cb or engine in ('lgb', 'xgb')) and not numeric_only else []
    txt = None
    if use_text and TXT_TR is not None:
        base_X = base_X.copy(); base_T = base_T.copy()
        base_X['txt_position'] = TXT_TR.values; base_T['txt_position'] = TXT_TE.values
        txt = ['txt_position']
    yy = np.log1p(y) if log_target else y
    hsh = _cfg_hash(engine, params, DEC_SIG, log_target, use_cb, use_text, numeric_only)
    oof = np.zeros(len(X)); te_pred = np.zeros(len(Xt))
    imp_total = None; n_runs = 0; fold_scores = []; n_cached = 0
    for seed in seeds:
        for k, (tr_i, va_i) in enumerate(SPLITS[seed]):
            ck = CACHE_DIR / f'{name}_{seed}_{k}_{hsh}.npz'
            gi = cols = None
            if USE_CACHE and ck.exists():
                z = np.load(ck, allow_pickle=False)
                pv, pt = z['pv'], z['pt']; n_cached += 1
                if 'imp' in z.files and z['imp'].size > 1:
                    gi, cols = z['imp'], z['cols']
            else:
                te_tr, te_va = te_for_fold(seed, k, tr_i, va_i)
                Xtr = pd.concat([base_X.iloc[tr_i], te_tr], axis=1)
                Xva = pd.concat([base_X.iloc[va_i], te_va], axis=1)
                Xte = pd.concat([base_T, TE_TEST], axis=1)
                pv, pt, m = fit_predict(engine, params, seed * 1000 + k,
                                        Xtr, yy[tr_i], W_TRAIN[tr_i],
                                        Xva, yy[va_i], w[va_i], Xte, cats, txt)
                if log_target:
                    pv, pt = np.expm1(pv), np.expm1(pt)
                if collect_imp and hasattr(m, 'booster_'):
                    gi = m.booster_.feature_importance('gain')
                    cols = np.array([str(c) for c in Xtr.columns])
                np.savez_compressed(ck, pv=pv, pt=pt,
                                    imp=(gi if gi is not None else np.zeros(1)),
                                    cols=(cols if cols is not None else np.array(['_'])))
            oof[va_i] += pv; te_pred += pt; n_runs += 1
            fold_scores.append(wmae(y[va_i], pv, w[va_i]))
            if collect_imp and gi is not None:
                s_ = pd.Series(gi, index=cols)
                imp_total = s_ if imp_total is None else imp_total.add(s_, fill_value=0.0)
    oof /= len(seeds); te_pred /= n_runs
    sc = wmae(y, oof, w)
    sc_late = wmae(y[LATE_MASK], oof[LATE_MASK], w[LATE_MASK])
    tag = f' | кэш {n_cached}/{n_runs}' if n_cached else ''
    print(f'[{name:9s}] OOF WMAE = {sc:9.2f} | посл.2мес = {sc_late:9.2f} | '
          f'folds ±{np.std(fold_scores):7.2f} | {time.time()-t0:7.1f}s{tag}')
    return {'oof': oof, 'test': te_pred, 'score': sc, 'score_late': sc_late,
            'imp': (imp_total / max(n_runs, 1) if imp_total is not None else None)}


In [ ]:
# 9b. ТЮНИНГ (Optuna, форвард-холдаут)
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False
    print('ВНИМАНИЕ: optuna не установлена (pip install optuna) -> использую дефолтные параметры')

FWD_TR = np.where(hmask_tr)[0]
FWD_VA = np.where(hmask_va)[0]
TE_FWD = te_for_pair(FWD_TR, FWD_VA, key='fwd')

def _fwd_eval(engine, params):
    te_tr, te_va = TE_FWD
    bX = X_cb if engine == 'cb' else X
    Xtr = pd.concat([bX.iloc[FWD_TR], te_tr], axis=1)
    Xva = pd.concat([bX.iloc[FWD_VA], te_va], axis=1)
    pv, _, _ = fit_predict(engine, params, SEED, Xtr, y[FWD_TR], W_TRAIN[FWD_TR],
                           Xva, y[FWD_VA], w[FWD_VA], Xva.head(1), CATS, None)
    return wmae(y[FWD_VA], pv, w[FWD_VA])

def tune_or_load(tag, n_trials, objective, default_params):
    path = ART_DIR / f'tuned_{tag}.json'
    if USE_CACHE and path.exists():
        saved = json.loads(path.read_text())
        print(f'[{tag}] параметры из кэша (WMAE на форвард-холдауте {saved.get("wmae", float("nan")):.2f}): '
              f'{saved["params"]}')
        return saved['params']
    if not HAS_OPTUNA or n_trials <= 0:
        return default_params
    t0 = time.time()
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    path.write_text(json.dumps({'params': study.best_params, 'wmae': study.best_value}))
    print(f'[{tag}] {n_trials} трейлов за {(time.time()-t0)/60:.1f} мин | '
          f'лучший форвард-WMAE={study.best_value:.2f}\n  params: {study.best_params}')
    return study.best_params

def _lgb_obj(trial):
    p = dict(LGB_D)
    p.update(num_leaves=trial.suggest_int('num_leaves', 63, 255, log=True),
             min_child_samples=trial.suggest_int('min_child_samples', 20, 160, log=True),
             learning_rate=trial.suggest_float('learning_rate', 0.015, 0.06, log=True),
             colsample_bytree=trial.suggest_float('colsample_bytree', 0.45, 0.9),
             subsample=trial.suggest_float('subsample', 0.7, 1.0),
             reg_lambda=trial.suggest_float('reg_lambda', 0.5, 40.0, log=True),
             reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 8.0, log=True),
             max_bin=trial.suggest_categorical('max_bin', [127, 255]))
    p['n_estimators'] = min(p['n_estimators'], 60 if FAST_MODE else 15000)
    return _fwd_eval('lgb', p)

def _xgb_obj(trial):
    p = dict(XGB_D)
    p.update(max_depth=trial.suggest_int('max_depth', 6, 10),
             min_child_weight=trial.suggest_float('min_child_weight', 4, 64, log=True),
             learning_rate=trial.suggest_float('learning_rate', 0.02, 0.07, log=True),
             subsample=trial.suggest_float('subsample', 0.7, 1.0),
             colsample_bytree=trial.suggest_float('colsample_bytree', 0.45, 0.9),
             reg_lambda=trial.suggest_float('reg_lambda', 0.5, 40.0, log=True),
             reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 8.0, log=True))
    p['n_estimators'] = min(p['n_estimators'], 60 if FAST_MODE else 15000)
    return _fwd_eval('xgb', p)

def _cb_obj(trial):
    p = dict(CB_D)
    p.update(grow_policy=trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise']),
             depth=trial.suggest_int('depth', 6, 10),
             learning_rate=trial.suggest_float('learning_rate', 0.04, 0.12, log=True),
             l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 2.0, 14.0),
             min_data_in_leaf=trial.suggest_int('min_data_in_leaf', 20, 100),
             subsample=trial.suggest_float('subsample', 0.7, 1.0),
             rsm=trial.suggest_float('rsm', 0.6, 1.0))
    p['iterations'] = min(p['iterations'], 60 if FAST_MODE else 9000)
    return _fwd_eval('cb', p)

BEST_LGB = tune_or_load('lgb', TRIALS['lgb'], _lgb_obj, {})
BEST_XGB = tune_or_load('xgb', TRIALS['xgb'], _xgb_obj, {})
BEST_CB  = tune_or_load('cb',  TRIALS['cb'],  _cb_obj,  {})
TUNED_LGB = {**LGB_D, **BEST_LGB}
TUNED_XGB = {**XGB_D, **BEST_XGB}
TUNED_CB  = {**CB_D,  **BEST_CB}


In [ ]:
# 10. ОБУЧЕНИЕ АНСАМБЛЯ v3
RESULTS = {}
RESULTS['lgb_opt']  = run_model('lgb_opt',  'lgb', TUNED_LGB, collect_imp=True)
RESULTS['lgb_log']  = run_model('lgb_log',  'lgb', TUNED_LGB, log_target=True)
RESULTS['lgb_q60']  = run_model('lgb_q60',  'lgb', {**TUNED_LGB, 'objective': 'quantile', 'alpha': 0.60})
RESULTS['lgb_et']   = run_model('lgb_et',   'lgb', {**TUNED_LGB, 'extra_trees': True,
                                'colsample_bytree': min(0.95, TUNED_LGB.get('colsample_bytree', 0.65) + 0.15)})
RESULTS['lgb_lin']  = run_model('lgb_lin',  'lgb', {**TUNED_LGB, 'linear_tree': True,
                                'num_leaves': min(96, TUNED_LGB.get('num_leaves', 96)),
                                'reg_lambda': max(5.0, TUNED_LGB.get('reg_lambda', 5.0))},
                                numeric_only=True)
RESULTS['xgb_opt']  = run_model('xgb_opt',  'xgb', TUNED_XGB)
RESULTS['hgb_l1']   = run_model('hgb_l1',   'hgb', HGB_D, numeric_only=True)
RESULTS['ridge_log'] = run_model('ridge_log', 'ridge', {'alpha': 30.0},
                                 log_target=True, numeric_only=True)
RESULTS['cb_opt']   = run_model('cb_opt',   'cb',  TUNED_CB, use_cb=True)
_n_txt = 0 if TXT_TR is None else int((TXT_TR != '').sum())
if _n_txt >= 500:
    try:
        RESULTS['cb_text'] = run_model('cb_text', 'cb', TUNED_CB, use_cb=True, use_text=True,
                                       seeds=SEEDS[:1])
    except Exception as e:
        print(f'[cb_text  ] пропущен: {type(e).__name__}: {str(e)[:110]}')
else:
    print(f'[cb_text  ] пропущен: непустых должностей всего {_n_txt} (< 500)')

print(f'\nБейзлайн (константа = взвешенная медиана): '
      f'WMAE = {wmae(y, np.full(len(y), weighted_median(y, w)), w):.2f}')
print(pd.DataFrame({'OOF_WMAE': {m: r['score'] for m, r in RESULTS.items()},
                    'OOF_посл2мес': {m: r['score_late'] for m, r in RESULTS.items()}})
      .sort_values('OOF_посл2мес').round(2))
_best = min(r['score'] for r in RESULTS.values())
if not FAST_MODE and _best > 66000:
    print('\n!!! ВНИМАНИЕ: лучшая модель хуже 66k — что-то не так, НЕ сабмитьте результат, '
          'проверьте лог выше.')


In [ ]:
# 11. АНСАМБЛЬ v3: TIME-AWARE МЕТА-СЛОЙ
from scipy.optimize import minimize

LATE = LATE_MASK
MODEL_ORDER = list(RESULTS.keys())
P_oof  = np.column_stack([RESULTS[m]['oof']  for m in MODEL_ORDER])
P_test = np.column_stack([RESULTS[m]['test'] for m in MODEL_ORDER])
yL, wL, PL = y[LATE], w[LATE], P_oof[LATE]

def blend_loss(v):
    v = np.clip(v, 0, None); s = v.sum()
    return 1e18 if s <= 0 else wmae(yL, PL @ (v / s), wL)

rng = np.random.default_rng(SEED)
cands = [np.ones(P_oof.shape[1])] + [np.eye(P_oof.shape[1])[i] for i in range(P_oof.shape[1])] \
        + list(rng.dirichlet(np.ones(P_oof.shape[1]), size=15))
nm_v, nm_s = None, np.inf
for v0 in cands:
    r = minimize(blend_loss, v0, method='Nelder-Mead',
                 options={'maxiter': 600, 'xatol': 1e-4, 'fatol': 1e-3})
    if r.fun < nm_s:
        nm_s, nm_v = r.fun, np.clip(r.x, 0, None)
nm_v = nm_v / nm_v.sum()

def caruana(iters=600):
    wsum = float(wL.sum())
    counts = np.zeros(PL.shape[1])
    counts[int(np.argmin([RESULTS[m]['score_late'] for m in MODEL_ORDER]))] = 1
    cur = PL @ counts; tot = 1.0
    cur_s = wmae(yL, cur / tot, wL)
    for _ in range(iters):
        errs = np.sum(wL[:, None] * np.abs((cur[:, None] + PL) / (tot + 1) - yL[:, None]), axis=0) / wsum
        j = int(np.argmin(errs))
        if errs[j] >= cur_s - 1e-9:
            break
        counts[j] += 1; cur = cur + PL[:, j]; tot += 1; cur_s = errs[j]
    return counts / counts.sum(), cur_s

ca_v, ca_s = caruana()
best_v, sc_late, src = (nm_v, nm_s, 'Nelder-Mead') if nm_s <= ca_s else (ca_v, ca_s, 'Caruana greedy')
print(f'Оптимизатор весов (на последних 2 мес): {src} (NM={nm_s:.2f} vs Caruana={ca_s:.2f})')
print('Веса бленда:', {m: round(float(c), 3) for m, c in zip(MODEL_ORDER, best_v) if c > 0.001})

blend_oof, blend_test = P_oof @ best_v, P_test @ best_v
sc_late_single = min(RESULTS[m]['score_late'] for m in MODEL_ORDER)
if sc_late > sc_late_single:
    m_best = min(RESULTS, key=lambda m: RESULTS[m]['score_late'])
    best_v = np.array([1.0 if m == m_best else 0.0 for m in MODEL_ORDER])
    blend_oof, blend_test, sc_late = RESULTS[m_best]['oof'], RESULTS[m_best]['test'], sc_late_single
print(f'WMAE посл.2мес: лучшая одиночная = {sc_late_single:.2f} | бленд = {sc_late:.2f} | '
      f'(полный OOF бленда = {wmae(y, blend_oof, w):.2f})')

def cal_loss(ab):
    a_, b_ = ab
    if not (0.95 <= a_ <= 1.10 and -3000 <= b_ <= 3000):
        return 1e18
    return wmae(yL, a_ * blend_oof[LATE] + b_, wL)
grid = [(a, b) for a in np.arange(0.95, 1.101, 0.005) for b in np.arange(-3000, 3001, 250)]
a0, b0 = min(grid, key=cal_loss)
res = minimize(cal_loss, x0=[a0, b0], method='Nelder-Mead', options={'xatol': 1e-4, 'fatol': 1e-3})
_cand = res.x if res.fun < cal_loss((a0, b0)) else np.array([a0, b0])
a_cal, b_cal = (float(_cand[0]), float(_cand[1])) if cal_loss(_cand) < sc_late else (1.0, 0.0)
oof_cal  = a_cal * blend_oof + b_cal
test_cal = a_cal * blend_test + b_cal
sc_cal_late = wmae(yL, oof_cal[LATE], wL)
print(f'Калибровка (посл.2мес): a={a_cal:.4f}, b={b_cal:.1f} -> WMAE посл.2мес = {sc_cal_late:.2f}')

N_BINS = 5
BIN_EDGES = np.quantile(oof_cal, np.linspace(0, 1, N_BINS + 1))
BIN_EDGES[0], BIN_EDGES[-1] = -np.inf, np.inf
BIN_F = np.ones(N_BINS)
_binL = np.digitize(oof_cal[LATE], BIN_EDGES[1:-1])
for b_ in range(N_BINS):
    mask = _binL == b_
    if mask.sum() < 200:
        continue
    lo_, hi_ = 0.90, 1.15
    for _ in range(45):
        m1 = lo_ + 0.382 * (hi_ - lo_); m2 = lo_ + 0.618 * (hi_ - lo_)
        if wmae(yL[mask], oof_cal[LATE][mask] * m1, wL[mask]) < wmae(yL[mask], oof_cal[LATE][mask] * m2, wL[mask]):
            hi_ = m2
        else:
            lo_ = m1
    BIN_F[b_] = (lo_ + hi_) / 2
sc_bin_late = wmae(yL, oof_cal[LATE] * BIN_F[_binL], wL)
if sc_bin_late <= sc_cal_late - 25:
    oof_cal  = oof_cal  * BIN_F[np.digitize(oof_cal,  BIN_EDGES[1:-1])]
    test_cal = test_cal * BIN_F[np.digitize(test_cal, BIN_EDGES[1:-1])]
    sc_cal_late = sc_bin_late
    print(f'Бин-калибровка принята: множители {np.round(BIN_F, 3)} -> WMAE посл.2мес = {sc_cal_late:.2f}')
else:
    BIN_F = np.ones(N_BINS)
    print(f'Бин-калибровка не дала выигрыша ({sc_bin_late:.2f}) — пропускаем')

lo = float(np.min(y))
oof_fin = np.maximum(oof_cal, lo)
test_fin = np.maximum(test_cal, lo)
sc_fin = wmae(y, oof_fin, w)
sc_fin_late = wmae(yL, oof_fin[LATE], wL)

if DEC['trend_g'] != 0.0:
    test_fin = test_fin * TEST_TREND_MULT
    print(f"Тренд-поправка test: g={DEC['trend_g']*100:+.2f}%/мес")

print(f'\n========== ИТОГ: WMAE посл.2мес = {sc_fin_late:.2f} | полный OOF = {sc_fin:.2f} ==========')

_ok = True
for q_ in (0.25, 0.5, 0.75):
    r_ = float(np.quantile(test_fin, q_) / np.quantile(y, q_))
    if not (0.6 <= r_ <= 1.6):
        _ok = False
        print(f'!!! ВНИМАНИЕ: q{q_} прогноза на test = {r_:.2f} от train — распределение неправдоподобно.')
if not FAST_MODE and sc_fin > 63000:
    _ok = False
    print(f'!!! ВНИМАНИЕ: полный OOF {sc_fin:.0f} хуже уровня v1 (60302).')
print('Санитарная проверка:', 'ПРОЙДЕНА — можно сабмитить' if _ok else 'ПРОВАЛЕНА — НЕ САБМИТЬТЕ, разберитесь в логе')


In [ ]:
# 12. РАЗБОР КАЧЕСТВА (для защиты)
rep = pd.DataFrame({'y': y, 'p': oof_fin, 'w': w,
                    'month': train['dt_parsed'].dt.to_period('M').astype(str),
                    'decile': pd.qcut(y, 10, labels=False, duplicates='drop')})
rep['wae'] = rep['w'] * (rep['y'] - rep['p']).abs()

print('WMAE по месяцам train (стабильность во времени):')
bym = rep.groupby('month').apply(lambda g: g['wae'].sum() / g['w'].sum())
print(bym.round(1).to_string())

print('\nWMAE по децилям дохода (где модель ошибается сильнее):')
byd = rep.groupby('decile').apply(lambda g: g['wae'].sum() / g['w'].sum())
print(byd.round(1).to_string())
print('-> основная ошибка в верхних децилях — типично для дохода (тяжёлый хвост).')

print('\nТоп-25 признаков по gain (LightGBM, усреднено по фолдам):')
imp = (RESULTS['lgb_opt']['imp'] if RESULTS['lgb_opt']['imp'] is not None
       else pd.Series(dtype=float)).sort_values(ascending=False).head(25)
for nm, g in imp.items():
    orig = INV_SAFE.get(nm, nm)
    print(f'  {orig:55s} {g:14.0f} | {DESC.get(orig, "")[:75]}')


print('\n[v2] Квантили прогноза на test vs таргет train (контроль сдвига уровня):')
for q_ in (0.05, 0.25, 0.5, 0.75, 0.95, 0.99):
    qt = float(np.quantile(test_fin, q_)); qy = float(np.quantile(y, q_))
    print(f'  q{q_:<5}: test_pred={qt:>12,.0f} | train_y={qy:>12,.0f} | ratio={qt/qy:.3f}')


In [ ]:
# 13. SUBMISSION
pred_df = pd.DataFrame({'id': test[ID_COL].values, 'predict': test_fin})

if SAMPLE_SUB_PATH:
    ss = pd.read_csv(SAMPLE_SUB_PATH, sep=';', decimal=',')
    if set(ss['id']) == set(pred_df['id']):
        pred_df = ss[['id']].merge(pred_df, on='id', how='left')
        print('Порядок id выровнен по sample_submission.')
    else:
        print('ВНИМАНИЕ: id в sample_submission не совпадают с test — сохраняю в порядке test.csv')

assert pred_df['predict'].notna().all() and len(pred_df) == len(test)
pred_df.set_index('id').to_csv(SUBMISSION_PATH, sep=';', decimal=',')
print(f'Сохранено: {SUBMISSION_PATH} | строк: {len(pred_df)}')
print(pred_df.head(5).to_string(index=False))
print('\nПроверка чтения файла обратно:')
print(pd.read_csv(SUBMISSION_PATH, sep=';', decimal=',').head(3).to_string(index=False))


Порядок id выровнен по sample_submission.
Сохранено: submission.csv | строк: 73214
 id      predict
  0 50344.256545
  1 46823.068646
  3 29914.214158
  9 75008.123825
 11 43420.900590

Проверка чтения файла обратно:
 id      predict
  0 50344.256545
  1 46823.068646
  3 29914.214158


In [ ]:
# 4. АРТЕФАКТЫ (воспроизводимость)
np.save(ART_DIR / 'oof_final.npy', oof_fin)
pd.DataFrame({'id': train[ID_COL], **{m: RESULTS[m]['oof'] for m in MODEL_ORDER}}) \
  .to_csv(ART_DIR / 'oof_by_model.csv', index=False)
pd.DataFrame({'id': test[ID_COL], **{m: RESULTS[m]['test'] for m in MODEL_ORDER}}) \
  .to_csv(ART_DIR / 'test_by_model.csv', index=False)
meta = {'version': 'v3', 'blend_optimizer': src,
        'meta_fit_on': 'последние 2 месяца OOF (time-aware)',
        'blend_weights': {m: float(c) for m, c in zip(MODEL_ORDER, best_v)},
        'calibration': {'a': float(a_cal), 'b': float(b_cal), 'clip_lo': lo,
                        'bin_factors': [float(f) for f in BIN_F]},
        'anti_drift_decisions': DEC,
        'adversarial_auc': float(AUC_ADV),
        'holdout_experiments': {k: float(v) for k, v in RES_EXP.items()},
        'train_only_dropped': TRAIN_ONLY_COLS,
        'tuned_params': {'lgb': BEST_LGB, 'xgb': BEST_XGB, 'cb': BEST_CB},
        'oof_wmae': {m: float(RESULTS[m]['score']) for m in MODEL_ORDER},
        'oof_wmae_late': {m: float(RESULTS[m]['score_late']) for m in MODEL_ORDER},
        'oof_wmae_final': float(sc_fin), 'oof_wmae_final_late': float(sc_fin_late),
        'n_folds': N_FOLDS, 'seeds': SEEDS, 'profile': PROFILE,
        'n_features': int(X.shape[1]) + len(TE_COLS_OUT), 'fast_mode': FAST_MODE}
(ART_DIR / 'run_meta_v3.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
print(json.dumps(meta, ensure_ascii=False, indent=2))


In [ ]:
# 15. ВЕРСИИ (для инструкции запуска)
import sys, sklearn, scipy, lightgbm, xgboost, catboost
print('python     :', sys.version.split()[0])
for lib in (np, pd, sklearn, scipy, lightgbm, xgboost, catboost):
    print(f'{lib.__name__:11s}:', lib.__version__)
print(f'\nПотоков CPU: {N_THREADS}. GPU не используется: Radeon RX 9070 XT не поддерживает CUDA,')
print('а GPU-режимы LightGBM/XGBoost/CatBoost под Windows требуют NVIDIA. На Ryzen 7800X3D')
print('CPU-обучение (hist) для 77k строк x ~240 признаков — оптимальный и воспроизводимый режим.')

try:
    import optuna as _op
    print(f'optuna     : {_op.__version__}')
except Exception:
    print('optuna     : не установлена (тюнинг будет пропущен)')


## Тезисы для защиты (по критериям регламента)

**Работа с данными.** Формат `sep=';'`, `decimal=','`; 34 колонки восстановлены из строк в числа; даты парсятся устойчиво (ISO -> dd.mm.yyyy -> mixed). Пропуски не заполняем (нативная обработка NaN + «богатство данных» как сигнал). Автоматически удалены признаки, у которых источник отключён в test (first_salary_income: 11% train / 0% test) — молчаливый сдвиг популяции NaN-веток.

**Временной сдвиг — главная особенность кейса.** Train: янв–июн 2024, test: июл–ноя 2024, adversarial AUC = 0.978. Отсюда разрыв случайного OOF (~60k) и public LB (~77k) у всех команд. Ответ v3 — сквозная time-aware схема: гиперпараметры тюнятся на форвард-холдауте (ранние месяцы -> последние 2), веса ансамбля и калибровка подбираются на OOF последних 2 месяцев, анти-дрейф приёмы (удаление/ранк дрейфующих, adversarial-веса, тренд-множитель) принимаются только если улучшили «будущее» на холдауте — в нашем прогоне все были честно отвергнуты, и это тоже результат: данные не подтвердили пользу агрессивных вмешательств.

**История ошибки — материал для критерия «валидация».** v2 содержала leave-one-out target encoding; LOO-код обратимо содержит таргет строки, бустинг это декодировал, OOF рухнул с 60k до 110k, сабмит — 139.8k (хуже константы). Диагноз поставлен по важности признаков (te-работодателя — топ-1 с 10-кратным отрывом) и по деградации всех 10 моделей одновременно. Исправление: внутренний 5-fold OOF для train-части фолда + исключение почти уникального ИНН работодателя из TE + санитарный контроль распределения прогнозов перед сабмитом. Это живой пример, почему leakage-проверка — отдельный обязательный этап.

**Признаки.** Консолидация 8 источников зарплаты, нормировка на регион, тренды оборотов 3м/12м, утилизация лимитов БКИ, ~23 флага должности + «грейд», динамика зарплаты клиента, TE четырёх категорий (город/регион/отделение/должность) по inner-OOF схеме, с диагностикой стабильности TE по месяцам.

**Моделирование.** Все модели оптимизируют взвешенный L1 (sample_weight = w = целевая WMAE). Ансамбль намеренно разнороден по индуктивному смещению: 5 вариантов LightGBM (включая linear_tree — кусочно-линейные листья, умеющие экстраполировать за диапазон train), XGBoost, HistGB, CatBoost (+ текстовый режим по должности) и Ridge на log-доходе — линейный класс, ошибающийся иначе, чем деревья. Веса — Nelder-Mead против жадного Caruana на поздних месяцах; калибровки с зажатыми границами; каждый шаг принимается только при улучшении метрики на поздних месяцах.

**Анти-оверфит к лидерборду.** Сабмитов мало и они осмысленные; в финал отмечаем лучший проверенный (v1) и v3 после валидации. Private, вероятно, поздние месяцы — верим форвард-схеме, а не колебаниям public.

**Воспроизводимость.** Фиксированные сиды, пины версий, все артефакты (OOF/test всех моделей, веса, параметры, решения, причина и фикс инцидента v2) в `artifacts_v3/`, дисковый кэш обучений — прерывание и продолжение безопасны.
